# 🎤 Clone Your Own Voice (Free) — ClarityVoice

This notebook uses the free, open-source **XTTS-v2** model to make speech in **your** voice, from a short recording of you talking.

### How to use
1. **Turn on the free GPU:** menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**.
2. Run each cell **top to bottom** (click the ▶ on the left of each cell).
3. When asked, upload a **clear 15–30 second recording of your own voice** (WAV or MP3, no background noise — just you talking naturally).
4. Type what you want it to say, run the last cell, and it plays + downloads your cloned voice.

> First run downloads the model (~2 min). That's normal.

### 1. Check the GPU is on

In [ ]:
!nvidia-smi -L
# If this says 'command not found' or shows no GPU, go to Runtime > Change runtime type > T4 GPU.

### 2. Install the voice library (~1 min)

In [ ]:
!pip install -q coqui-tts "transformers==4.49.0"
print("Installed. Now do Runtime > Restart session, then run from cell 3 (skip this install).")

### 3. Load the model

In [ ]:
import os, torch
os.environ["COQUI_TOS_AGREED"] = "1"

# Make XTTS load on new PyTorch. Patch ONCE (safe to re-run — avoids RecursionError):
if not getattr(torch, "_xtts_patched", False):
    _real_torch_load = torch.load
    def _safe_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _real_torch_load(*args, **kwargs)
    torch.load = _safe_load
    torch._xtts_patched = True

from TTS.api import TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("Model loaded ✅")

### 4. Upload YOUR voice recordings (2-4 clean clips, ~1-3 min total)
Quiet room, natural speaking, WAV or M4A. More clean audio = a clone that sounds more like you.
Avoid WhatsApp voice notes (too compressed).

In [ ]:
from google.colab import files
print("Upload 2-4 CLEAN recordings of you (hold Ctrl to select multiple). Total ~1-3 minutes.")
print("Quiet room, natural speaking, phone close. WAV or M4A. Avoid WhatsApp voice notes.")
uploaded = files.upload()
sample_paths = list(uploaded.keys())   # all clips blended = most stable, faithful clone
print(f"\nUsing {len(sample_paths)} file(s):", sample_paths)

### 5. Make it speak in your voice — multiple languages ✨
Each line below is one language. Edit the text, then run. This model supports **English (`en`)** and **Hindi (`hi`)**. Kannada is **not** supported here (ask for the Indic model for Kannada).

In [ ]:
from IPython.display import Audio, display

# Keep the text NATURAL and well-punctuated.
# (XTTS reads plain sentences best — fake fillers like "umm/haha/niiice" sound robotic in this model.)
clips = {
    "en": "Hello, this is my own voice. I think it sounds clear and natural.",
    "hi": "नमस्ते, यह मेरी अपनी आवाज़ है। मुझे लगता है यह साफ़ और स्वाभाविक लगती है।",
}

for lang, text in clips.items():
    out = f"my_voice_{lang}.wav"
    tts.tts_to_file(
        text=text, speaker_wav=sample_paths, language=lang, file_path=out,
        temperature=0.7,          # lower = more faithful to YOUR voice + fewer artifacts
        repetition_penalty=2.0,
        top_k=50, top_p=0.8,
        length_penalty=1.0,
        speed=1.0,
        enable_text_splitting=True,
    )
    print(f"\n=== {lang} ===")
    display(Audio(out))
    files.download(out)

print("\nTuned for fidelity (sounds like you). Big expressive emotion is beyond free XTTS —")
print("for that, fine-tuning (free, heavy) or ElevenLabs (paid, easy) is the real step up.")